In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from pygraphon.graphons import StepGraphon
from pygraphon.estimators import SmoothNetHist

from utils_generators import AssociativeFullyRandom
from pygraphon.norm import MseProbaEdge, MaeProbaEdge, AUCEdge, AUPRCEdge
import matplotlib.pyplot as plt
from pygraphon.plots.plot_matrices import spy


# Initialisation

In [ ]:
n = 100
k = 2
theta_intra_min, theta_intra_max = 0.5, 0.7
theta_inter_min, theta_inter_max = 0.05, 0.2
theta = AssociativeFullyRandom(
                    K=k,
                    min_inside_classes=theta_intra_min,
                    max_inside_classes=theta_intra_max,
                    min_outside_classes=theta_inter_min,
                    max_outside_classes=theta_inter_max,
                )
print(theta.Theta)
graphon = StepGraphon(theta.Theta, bandwidthHist=1 / k)
adjacency_matrix = graphon.draw(n=n, exchangeable=False)

# Olhede fit

In [ ]:
est = SmoothNetHist()
first_hist_olhede = est._first_approximate_graphon_from_adjacency(adjacencyMatrix= adjacency_matrix)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im2 = ax[2].imshow(first_hist_olhede[0].get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
_ = spy(adjacency_matrix, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("True graphon")
ax[2].set_title("Olhede histogram")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)
plt.show()

In [ ]:
from pygraphon.utils import edge_density
spy(adjacency_matrix)
print("Edge density: ", edge_density(adjacency_matrix[int(n/2):, int(n/2):]))
print("Edge density: ", edge_density(adjacency_matrix[:int(n/2), :int(n/2)]))
print("Edge density: ", edge_density(adjacency_matrix[:int(n/2), int(n/2):]))

# Smooth NetHist fit 

In [ ]:
my_est = SmoothNetHist()
my_est.fit(adjacency_matrix)

In [ ]:
my_est._first_fit_graphon

In [ ]:
print("Ratio of parameters:", my_est.get_ratio_par())
print("Number of unique parameters in SmoothNetHist:", len(np.unique(my_est.graphon.get_graphon())))
print("Number of unique parameters in Olhede:", len(np.unique(first_hist_olhede[0].get_graphon())))


In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im2 = ax[2].imshow(my_est._first_fit_graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im3 = ax[3].imshow(my_est.graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
_ = spy(adjacency_matrix, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("True graphon")
ax[2].set_title("Olhede histogram")
ax[3].set_title("SmoothNetHist")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)
plt.colorbar(im3, cax=cbar_ax)
plt.show()

In [ ]:
my_est._first_fit_graphon

In [ ]:
norm_olhede_mse = MseProbaEdge(n_nodes=n)(graphon, my_est._first_fit_graphon)
norm_olhede_mae = MaeProbaEdge(n_nodes=n)(graphon, my_est._first_fit_graphon)
norm_olhede_auc = AUCEdge(n_nodes=n)(adjacency_matrix, my_est._first_fit_graphon.get_edge_probabilities(n,False,True))
norm_olhede_auprc = AUPRCEdge(n_nodes=n)(adjacency_matrix, my_est._first_fit_graphon.get_edge_probabilities(n,False,True))

norm_smoothnet_mse = MseProbaEdge(n_nodes=n)(graphon, my_est.graphon)
norm_smoothnet_mae = MaeProbaEdge(n_nodes=n)(graphon, my_est.graphon)
norm_smoothnet_auc = AUCEdge(n_nodes=n)(adjacency_matrix, my_est.graphon.get_edge_probabilities(n,False,True))
norm_smoothnet_auprc = AUPRCEdge(n_nodes=n)(adjacency_matrix, my_est.graphon.get_edge_probabilities(n,False,True))

print("MSE Olhede vs SmoothNetHist:", norm_olhede_mse, norm_smoothnet_mse)
print("MAE Olhede vs SmoothNetHist:", norm_olhede_mae, norm_smoothnet_mae)
print("AUC Olhede vs SmoothNetHist:", norm_olhede_auc, norm_smoothnet_auc)
print("AUPRC Olhede vs SmoothNetHist:", norm_olhede_auprc, norm_smoothnet_auprc)

# SmoothNetHist step by step

In [ ]:
new_est = SmoothNetHist()
finite_est = new_est._approximate_graphon_from_adjacency(adjacencyMatrix= adjacency_matrix, number_link_communities= int((k+1)*k/2))


In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im2 = ax[2].imshow(my_est._first_fit_graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im3 = ax[3].imshow(my_est.graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im4 = ax[4].imshow(finite_est[0].get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
_ = spy(adjacency_matrix, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("True graphon")
ax[2].set_title("Olhede histogram")
ax[3].set_title("SmoothNetHist")
ax[4].set_title("SmoothNetHist finite")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)
plt.colorbar(im3, cax=cbar_ax)
plt.colorbar(im4, cax=cbar_ax)
plt.show()

In [ ]:
norm_smoothnet_mse_true = MseProbaEdge(n_nodes=n)(graphon, finite_est[0])
norm_smoothnet_mae_true = MaeProbaEdge(n_nodes=n)(graphon, finite_est[0])
norm_smoothnet_auc_true = AUCEdge(n_nodes=n)(adjacency_matrix, finite_est[0].get_edge_probabilities(n,False,True))
norm_smoothnet_auprc_true = AUPRCEdge(n_nodes=n)(adjacency_matrix, finite_est[0].get_edge_probabilities(n,False,True))

print("MSE Olhede vs SmoothNetHist vs Corrected:", norm_olhede_mse, norm_smoothnet_mse, norm_smoothnet_mse_true)
print("MAE Olhede vs SmoothNetHist vs Corrected:", norm_olhede_mae, norm_smoothnet_mae, norm_smoothnet_mae_true)
print("AUC Olhede vs SmoothNetHist vs Corrected:", norm_olhede_auc, norm_smoothnet_auc, norm_smoothnet_auc_true)
print("AUPRC Olhede vs SmoothNetHist vs Corrected:", norm_olhede_auprc, norm_smoothnet_auprc, norm_smoothnet_auprc_true)

# Other plots for paper

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im2 = ax[2].imshow(my_est._first_fit_graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im3 = ax[3].imshow(my_est.graphon.get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
im4 = ax[4].imshow(finite_est[0].get_edge_probabilities(n,False,True), cmap='jet', vmin=0, vmax=1)
_ = spy(adjacency_matrix, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("True graphon")
ax[2].set_title("Olhede histogram")
ax[3].set_title("SmoothNetHist")
ax[4].set_title("SmoothNetHist finite")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)
plt.colorbar(im3, cax=cbar_ax)
plt.colorbar(im4, cax=cbar_ax)
plt.show()

# Polblog

In [ ]:
import io
from zipfile import ZipFile
import requests
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

url =  "https://networks.skewed.de/net/polblogs/files/polblogs.csv.zip"
response = requests.get(url, stream=False)
with ZipFile(io.BytesIO(response.content)) as myzip:
   with myzip.open(myzip.namelist()[0]) as myfile:
      df = pd.read_csv(myfile)
   with myzip.open(myzip.namelist()[1]) as myfile:
      df2 = pd.read_csv(myfile)

df.columns = ["source", "target"]
df2.columns = [s.replace(" ","") for s in df2.columns]
z = df2["value"].values

   
#df2.columns = ["value"] # 0 for liberals, 1 for conservative
G = nx.from_pandas_edgelist(df, "source", "target")
A = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))
# we remove the diagonal to avoid self-loops
A -= np.diag(np.diag(A))
spy(A)

In [ ]:
for index, row in df.iterrows():
    u = row["source"]
    v = row["target"] 
    if not G.has_edge(u, v):
        print("missing edge", u, v)
    if not G.has_edge(v, u):
        print("missing edge", v, u)

In [ ]:
#To check if Sofia is correct also
import networkx as nx
import numpy as np
from pygraphon.plots.plot_matrices import spy

# Load the edge list into a NetworkX graph
G_txt = nx.read_edgelist("polblogs.txt")

# Convert the graph into an adjacency matrix
adjacency_matrix = nx.to_numpy_array(G_txt)

# If you want to represent the adjacency matrix as a NumPy array:
A_olhede = np.array(adjacency_matrix)
A_olhede -= np.diag(np.diag(A_olhede))

In [ ]:
num_liberals = np.sum(z == 0)
num_conservatives = np.sum(z == 1)
print(num_conservatives)
print(num_liberals)

Trying to replicate the plots by sorting into 4 categories

In [ ]:
import networkx as nx
from random import shuffle

# Compute the degrees of the nodes
degrees = dict(G.degree())

def check_cross_citation(graph,node):
    neighbors = graph.neighbors(node)
    for neighbor in neighbors:
        if z[node] != z[neighbor]:
            return True
    return False

# Sort nodes by increasing degrees within each membership group
nodes_0_0 = [node for node in G.nodes() if z[node] == 0 and not check_cross_citation(G,node)]
nodes_0_1 = [node for node in G.nodes() if z[node] == 0 and check_cross_citation(G,node)]
nodes_1_0 = [node for node in G.nodes() if z[node] == 1 and  check_cross_citation(G,node)]
nodes_1_1 = [node for node in G.nodes() if z[node] == 1 and not check_cross_citation(G,node)]

shuffle(nodes_0_0)
shuffle(nodes_1_1)

nodes_0_0_sorted = nodes_0_0 #sorted(nodes_0_0, key=lambda node: degrees[node])
nodes_0_1_sorted = sorted(nodes_0_1, key=lambda node: degrees[node])
nodes_1_0_sorted = sorted(nodes_1_0, key=lambda node: degrees[node], reverse=True)
nodes_1_1_sorted = nodes_1_1 # sorted(nodes_1_1, key=lambda node: degrees[node], reverse=False)

# Create the reordered adjacency matrix
reordered_A = nx.adjacency_matrix(G, nodelist=nodes_0_0_sorted + nodes_0_1_sorted + nodes_1_0_sorted + nodes_1_1_sorted).toarray()

spy(reordered_A, marker_size=1, origin="lower")
plt.show()

## Trying to sort by degree of connectivity to other category only 

In [ ]:
# reload data to make sure no pollution from previous manipulations
G_gml = nx.read_gml("polblogs.gml", destringizer=int).to_undirected(reciprocal=False, as_view=False)

G_gml.remove_nodes_from(list(nx.isolates(G_gml)))

# Compute the number of connection to the opposite party
def count_cross_connections(graph,node) -> int:
    neighbors = graph.neighbors(node[0])
    count = 0
    for neighbor in neighbors:
        if node[1] != graph.nodes[neighbor]["value"]:
            count += 1
    return count

inter_connections ={node[0]: count_cross_connections(G_gml,node) for node in G_gml.nodes(data="value")}

# Sort nodes by increasing degrees within each membership group
nodes_0 = [node[0] for node in G_gml.nodes(data="value") if node[1] == 0 ]
nodes_1 = [node[0]  for node in G_gml.nodes(data="value") if node[1] == 1]

nodes_0_sorted = sorted(nodes_0, key=lambda node: inter_connections[node], reverse=False)
nodes_1_sorted = sorted(nodes_1, key=lambda node: inter_connections[node], reverse=True)

# Create the reordered adjacency matrix
reordered_A_gml = nx.adjacency_matrix(G_gml, nodelist=nodes_0_sorted  + nodes_1_sorted).toarray()
reordered_A_gml -= np.diag(np.diag(reordered_A_gml))
reordered_A_gml[reordered_A_gml == 2] = 1
print(np.unique(reordered_A_gml))
print(np.argwhere(reordered_A_gml == 2))
spy(reordered_A_gml, marker_size=0.5, origin="lower")
plt.show()

In [ ]:
from pygraphon import HistogramEstimator

est = SmoothNetHist()
est.fit(reordered_A_gml)
first_hist_olhede = est._first_approximate_graphon_from_adjacency(adjacencyMatrix= reordered_A_gml)
est_origin = HistogramEstimator()
est_origin.fit(reordered_A_gml)

print(est.get_ratio_par())
print("Number of unique parameters in SmoothNetHist:", len(np.unique(est.graphon.get_graphon())))
print("Number of unique parameters in Olhede:", len(np.unique(est_origin.get_graphon().graphon)))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(est_origin.get_edge_connectivity(), cmap='jet')#, vmin=0, vmax=1)
im2 = ax[2].imshow(est.get_edge_connectivity(), cmap='jet')#, vmin=0, vmax=1)
_ = spy(reordered_A_gml, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("Olhede histogram")
ax[2].set_title("SmoothNetHist")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(np.sqrt(est_origin.get_graphon().graphon), cmap='jet')#, vmin=0, vmax=1)
im2 = ax[2].imshow(np.sqrt(est.get_graphon().graphon), cmap='jet')#, vmin=0, vmax=1)
_ = spy(reordered_A_gml, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("Olhede histogram")
ax[2].set_title("SmoothNetHist")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)

plt.show()

In [ ]:
def print_array_with_indices(arr):
    for index, value in np.ndenumerate(arr):
        print(f"Index: {index}, Value: {value}")

print_array_with_indices(np.diag(est_origin.graphon.get_graphon()))

In [ ]:
turn_index = 8
end_index = est_origin.graphon.get_graphon().shape[0]
sort_index_dem = [turn_index - (i+1) for i in range(turn_index) ] + [end_index - (i+1) for i in range(end_index - turn_index)]
print(sort_index_dem)

In [ ]:
graphon_polblogs = est.graphon.get_graphon()
# Get the diagonal elements
diagonal = np.diag(graphon_polblogs)

# Sort the diagonal elements in increasing order and get the sorting indices
sorting_indices = np.argsort(diagonal)
print(sorting_indices)

# Permute the rows and columns of the matrix based on the sorting indices
graphon_polblogs_permuted = graphon_polblogs[sort_index_dem][:, sort_index_dem]

print(graphon_polblogs_permuted.shape)

In [ ]:
fig = plt.figure(figsize=(6, 3.2))

ax = fig.add_subplot(111)
ax.set_title('colorMap')
plt.imshow(graphon_polblogs_permuted, cmap='jet')
ax.set_aspect('equal')

cax = fig.add_axes([0.12, 0.1, 0.78, 0.8])
cax.get_xaxis().set_visible(False)
cax.get_yaxis().set_visible(False)
cax.patch.set_alpha(0)
cax.set_frame_on(False)
plt.colorbar(orientation='vertical')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_3d_and_reflection(arr):
    # Create a colormap for visualization
    cmap = plt.get_cmap('jet')

    # 3D Plot
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    # Generate coordinates for the bars
    x_pos, y_pos = np.meshgrid(np.arange(arr.shape[1]), np.arange(arr.shape[0]))
    x_pos = x_pos.flatten()
    y_pos = y_pos.flatten()
    z_pos = np.zeros_like(x_pos)

    # Generate the heights of the bars proportional to the array values
    dx = dy = 0.8
    dz = arr.flatten()
    
    # Plot the 3D bars
    ax.bar3d(x_pos, y_pos, z_pos, dx, dy, dz, shade=True, color=cmap(arr.flatten() / arr.max()))

    # Set labels and title for the 3D plot
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('3D Plot with Color and Height')

    # 2D Reflection Plot
    fig2, ax2 = plt.subplots(figsize=(8, 6))
    
    # Plot the 2D array with colors
    im = ax2.imshow(arr, cmap=cmap)
    
    # Create a colorbar for the 2D plot
    cbar = plt.colorbar(im, ax=ax2)
    cbar.set_label('Value')

    # Set labels and title for the 2D plot
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_title('2D Reflection Plot with Color')

    # Show the plots
    plt.show()

# Example 2D numpy array
array_data = np.array([[1, 2, 3],
                       [4, 5, 6],
                       [7, 8, 9]])

# Plot the 3D plot and its reflection
plot_3d_and_reflection(np.sqrt(graphon_polblogs_permuted))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.gridspec as gridspec

def plot_3d_and_reflection_stacked(arr):
    # Create a colormap for visualization
    cmap = plt.get_cmap('jet')

    # Create a figure with a grid layout
    fig = plt.figure(figsize=(8, 10))
    gs = gridspec.GridSpec(2, 1, height_ratios=[1, 1])

    # 3D Plot
    ax_3d = fig.add_subplot(gs[0], projection='3d')

    # Generate coordinates for the bars
    x_pos, y_pos = np.meshgrid(np.arange(arr.shape[1]), np.arange(arr.shape[0]))
    x_pos = x_pos.flatten()
    y_pos = y_pos.flatten()
    z_pos = np.zeros_like(x_pos)

    # Generate the heights of the bars proportional to the array values
    dx = dy = 0.8
    dz = arr.flatten()
    
    # Plot the 3D bars
    ax_3d.bar3d(x_pos, y_pos, z_pos, dx, dy, dz, shade=True, color=cmap(arr.flatten() / arr.max()))

    # Set labels and title for the 3D plot
    ax_3d.set_xlabel('X')
    ax_3d.set_ylabel('Y')
    ax_3d.set_zlabel('Z')
    ax_3d.set_title('3D Plot with Color and Height')

    # 2D Reflection Plot
    ax_2d = fig.add_subplot(gs[1], projection='3d')
    
    # Plot the 3D bars
    ax_2d.bar3d(x_pos, y_pos, z_pos, dx, dy, 0, shade=True, color=cmap(arr.flatten() / arr.max()))

    # Set labels and title for the 2D plot
    ax_2d.set_xlabel('X')
    ax_2d.set_ylabel('Y')
    ax_2d.set_title('2D Reflection Plot with Color')

    # Adjust the layout to prevent clipping of labels and titles
    plt.tight_layout()

    # Show the plot
    plt.show()

# Example 2D numpy array
array_data = np.array([[1, 2, 3],
                       [4, 5, 6],
                       [7, 8, 9]])

# Plot the 3D plot and its reflection stacked and aligned
plot_3d_and_reflection_stacked(graphon_polblogs_permuted)
print(graphon_polblogs_permuted)


In [ ]:
import sys
sys.path.insert(0, r"c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages")
from mpl3d import glm
from mpl3d.camera import Camera
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from new_plots import Bar
Z = graphon_polblogs_permuted
print(Z)
cmap = plt.get_cmap("Reds")
norm = mpl.colors.Normalize(vmin=0,vmax=1)
facecolors = cmap(norm(Z))

fig = plt.figure(figsize=(10,5))
ax = fig.add_axes([0,0,1,1], xlim=[-1,+1], ylim=[0,1], aspect=1)
ax.axis("off")

camera = Camera("perspective", 65, -135)
bars = Bar(ax, camera.transform, Z, facecolors=facecolors)
camera.connect(ax, bars.update)

plt.savefig("bar.png", dpi=300)
plt.show()


# Gradient like polblog

In [ ]:
# Compute the degrees of the nodes
degrees = dict(G.degree())
nodes_0_sorted = sorted(G.nodes(), key=lambda node: degrees[node], reverse=True)
# Create the reordered adjacency matrix
reordered_A = nx.adjacency_matrix(G, nodelist=nodes_0_sorted).toarray()
reordered_A -= np.diag(np.diag(reordered_A))
spy(reordered_A, marker_size=1,  ds="default", origin = "lower")
plt.show()

In [ ]:
est = SmoothNetHist()
est.fit(reordered_A)
first_hist_olhede = est._first_approximate_graphon_from_adjacency(adjacencyMatrix= reordered_A)

In [ ]:
from pygraphon import HistogramEstimator
est_origin = HistogramEstimator()
est_origin.fit(reordered_A)

In [ ]:
print(est.get_ratio_par())
print("Number of unique parameters in SmoothNetHist:", len(np.unique(est.graphon.get_graphon())))
print("Number of unique parameters in Olhede:", len(np.unique(est_origin.get_graphon().graphon)))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(est_origin.get_edge_connectivity(), cmap='jet')#, vmin=0, vmax=1)
im2 = ax[2].imshow(est.get_edge_connectivity(), cmap='jet')#, vmin=0, vmax=1)
_ = spy(reordered_A, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("Olhede histogram")
ax[2].set_title("SmoothNetHist")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)

plt.show()

In [ ]:
plt.imshow(reordered_A,  interpolation = "none")

In [ ]:

fig, ax = plt.subplots(1, 3, figsize=(10, 5))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.95, 0.15, 0.05, 0.7])
im1 = ax[1].imshow(np.sqrt(est_origin.get_graphon().graphon), cmap='jet')
im2 = ax[2].imshow(np.sqrt(est.get_graphon().graphon), cmap='jet')
_ = spy(reordered_A, ax=ax[0])
ax[0].set_title("Adjacency matrix")
ax[1].set_title("Olhede histogram")
ax[2].set_title("SmoothNetHist")
plt.colorbar(im1, cax=cbar_ax)
plt.colorbar(im2, cax=cbar_ax)

plt.show()

In [ ]:
import numpy as np
np.diag(np.sqrt(est.get_graphon().graphon))

In [ ]:
graphon_polblogs = est.graphon.get_graphon()
# Get the diagonal elements
diagonal = np.diag(graphon_polblogs)

# Sort the diagonal elements in increasing order and get the sorting indices
sorting_indices = np.argsort(diagonal)

# Permute the rows and columns of the matrix based on the sorting indices
graphon_polblogs_permuted = graphon_polblogs[sorting_indices][:, sorting_indices]

print(graphon_polblogs_permuted.shape)

In [ ]:
fig = plt.figure(figsize=(6, 3.2))

ax = fig.add_subplot(111)
ax.set_title('colorMap')
plt.imshow(graphon_polblogs_permuted, cmap='jet')
ax.set_aspect('equal')

cax = fig.add_axes([0.12, 0.1, 0.78, 0.8])
cax.get_xaxis().set_visible(False)
cax.get_yaxis().set_visible(False)
cax.patch.set_alpha(0)
cax.set_frame_on(False)
plt.colorbar(orientation='vertical')
plt.show()